# Clustering de productos por forma de serie (DTW)

Viene de `01_EDA_series.ipynb`, que dejó planteada la pregunta con un número concreto:

```
correlación media entre pares de la MISMA cat3   : +0,327
correlación media entre pares de DISTINTA cat3   : +0,230
```

Apenas 0,1 de diferencia. **La jerarquía comercial casi no describe el comportamiento
temporal.** Si se pudiera agrupar productos por *cómo se mueven* en vez de por qué
son, ese agrupamiento le daría al modelo información que hoy no tiene.

## Por qué DTW y no correlación

La correlación compara mes contra mes: si dos productos tienen la misma forma pero
uno la vive dos meses más tarde, la correlación los ve distintos. **Dynamic Time
Warping** permite estirar y comprimir el eje temporal, así que reconoce la forma
aunque esté desfasada. Para ciclos de vida — donde lo que importa es *subió, hizo
pico, cayó* y no *en qué mes exacto* — es la métrica adecuada.

El `window` (banda de Sakoe-Chiba) limita cuánto se puede deformar el tiempo: sin
límite, DTW encuentra parecidos absurdos alineando un mes con otro a dos años de
distancia.

## Dos preguntas distintas, dos ejes

| `EJE` | series alineadas por | agrupa por | sirve para |
|---|---|---|---|
| `calendario` | mes real | comportamiento simultáneo (estacionalidad, shocks comunes) | features de predicción |
| `edad` | meses desde el lanzamiento | forma del ciclo de vida | entender arquetipos de producto |

> **Sobre leakage**: el cluster se calcula con la serie histórica, así que si se usa
> toda la historia hasta 201912 y después se valida prediciendo 201910, la etiqueta
> ya vio el futuro. Por eso hay un `MES_CORTE`: el clustering usa **sólo meses
> anteriores al corte**, el mismo que separa train de validación en `03_Optuna`.

## 0 — Ambiente y paleta

In [1]:
import os, json
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score


# Para correr local en Windows
os.environ["LABO3_BUCKET"] = r"C:\Users\Natalia\labo3-bucket"


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        return Path(env).expanduser().resolve()

    for cand in ("/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)

    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "datasets_fe"
DIR_OUT.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ   = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})

def limpiar(ax, titulo=None, y=None, x=None):
    if titulo:
        ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y:
        ax.set_ylabel(y)
    if x:
        ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax

print(f"BUCKET: {BUCKET}")
print(f"DTW con backend C: {dtw.try_import_c()}")

BUCKET: C:\Users\Natalia\labo3-bucket
DTW con backend C: True


## 1 — Palancas

In [4]:
PARAM = {
    # Eje temporal: 'calendario' (comportamiento simultaneo) | 'edad' (forma del ciclo)
    'eje': 'calendario',

    # ── ANTI-LEAKAGE ─────────────────────────────────────────────────────
    # Solo se usan meses ESTRICTAMENTE ANTERIORES a este corte para clusterizar.
    # Debe coincidir con el ultimo mes de train de 03_Optuna (default alli: 201905).
    # None = usar toda la historia (SOLO para exploracion, nunca para entrenar).
    'mes_corte': 201906,

    # Minimo de meses observados para que un producto entre al clustering
    'min_meses': 18,

    # Banda de Sakoe-Chiba: cuantos meses de desfase se permite al alinear
    'window': 6,

    # Normalizacion de cada serie antes de comparar formas
    # 'zscore' -> forma pura (ignora escala) | 'pico' -> 0..1 contra su maximo
    'normalizacion': 'zscore',

    # Rango de k a evaluar
    'k_min': 2, 'k_max': 10,

    # Metodo de linkage. 'ward' es el default por evidencia, no por gusto: la
    # seccion 5 compara los tres y muestra que 'average' y 'complete' desprenden
    # outliers de a uno y dejan un cluster gigante inservible.
    'linkage': 'ward',

    # Tamanio minimo de cluster como fraccion del total. Un k que deja un cluster
    # de 3 productos no sirve ni como feature categorica ni para modelo propio.
    'min_frac_cluster': 0.05,

    # NUMERO DE CLUSTERS. Se fija a mano a proposito: en estos datos el silhouette
    # decrece monotonamente con k, asi que "optimizarlo" siempre devuelve k=2, que
    # es tan poco informativo como cat3. Ver la discusion de la seccion 5.
    # None = usar el sugerido por silhouette entre los balanceados.
    'k': 6,
}
print(PARAM)

{'eje': 'calendario', 'mes_corte': 201906, 'min_meses': 18, 'window': 6, 'normalizacion': 'zscore', 'k_min': 2, 'k_max': 10, 'linkage': 'ward', 'min_frac_cluster': 0.05, 'k': 6}


## 2 — Panel producto-mes

In [5]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])

def a_m(col):
    return (pl.col(col) // 100) * 12 + (pl.col(col) % 100)

panel = (sell.group_by(["product_id", "periodo"]).agg(pl.col("tn").sum().alias("tn"))
             .with_columns(a_m("periodo").alias("m")).sort(["product_id", "m"]))

# Corte anti-leakage
if PARAM['mes_corte'] is not None:
    m_corte = (PARAM['mes_corte'] // 100) * 12 + (PARAM['mes_corte'] % 100)
    antes = panel.height
    panel = panel.filter(pl.col("m") < m_corte)
    print(f"CORTE en {PARAM['mes_corte']}: {antes:,} -> {panel.height:,} filas "
          f"(se descartan {antes-panel.height:,} del futuro)")
else:
    print("SIN CORTE: se usa toda la historia. Valido para explorar, NO para entrenar.")

# Densificar dentro de la vida de cada producto
vida = panel.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"),
    pl.col("tn").sum().alias("tn_total"), pl.len().alias("n_obs"))

grilla = (vida.select("product_id", "m_nace", "m_muere")
              .with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").drop("m_nace", "m_muere"))

panel = (grilla.join(panel, on=["product_id", "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .join(vida, on="product_id", how="left")
               .with_columns((pl.col("m") - pl.col("m_nace")).alias("edad"),
                             (pl.col("m_muere") - pl.col("m_nace") + 1).alias("largo"))
               .join(prod.select("product_id","cat1","cat2","cat3","brand"),
                     on="product_id", how="left")
               .sort(["product_id", "m"]))

M_MIN, M_MAX = panel["m"].min(), panel["m"].max()
def m_a_periodo(m): return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1
print(f"panel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")
print(f"rango: {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}")

CORTE en 201906: 31,243 -> 24,718 filas (se descartan 6,525 del futuro)
panel: 24,930 filas · 1119 productos
rango: 201701 -> 201905


C:\Users\Natalia\AppData\Local\Temp\ipykernel_43876\3240351428.py:27: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("m").drop("m_nace", "m_muere"))


## 3 — Series normalizadas

Una lista de arrays, uno por producto. DTW **acepta series de distinto largo** — no
hay que rellenar ni recortar, que es otra ventaja sobre la correlación.

In [6]:
elegibles = vida.filter((pl.col("m_muere") - pl.col("m_nace") + 1) >= PARAM['min_meses'])
elegibles = elegibles.sort("tn_total", descending=True)
PIDS = elegibles["product_id"].to_list()
print(f"{len(PIDS)} productos con >= {PARAM['min_meses']} meses (de {vida.height})")

col_orden = "m" if PARAM['eje'] == "calendario" else "edad"
sub = panel.filter(pl.col("product_id").is_in(PIDS)).sort(["product_id", col_orden])

SERIES, PIDS_OK = [], []
for pid, g in sub.group_by("product_id", maintain_order=True):
    pid = pid[0] if isinstance(pid, tuple) else pid
    v = g["tn"].to_numpy().astype(np.double)
    if PARAM['normalizacion'] == "zscore":
        sd = v.std()
        v = (v - v.mean()) / (sd if sd > 0 else 1.0)
    else:
        mx = v.max()
        v = v / (mx if mx > 0 else 1.0)
    if not np.all(np.isfinite(v)):
        continue
    SERIES.append(v); PIDS_OK.append(pid)

PIDS = PIDS_OK
largos = [len(s) for s in SERIES]
print(f"{len(SERIES)} series listas · largo min/mediano/max: "
      f"{min(largos)}/{int(np.median(largos))}/{max(largos)} meses")
print(f"eje: {PARAM['eje']}   normalizacion: {PARAM['normalizacion']}")

807 productos con >= 18 meses (de 1119)
807 series listas · largo min/mediano/max: 18/29/29 meses
eje: calendario   normalizacion: zscore


## 4 — Matriz de distancias DTW

In [ ]:
D = dtw.distance_matrix_fast(SERIES, window=PARAM['window'])
D = np.array(D)
# dtaidistance devuelve triangular superior con inf abajo: la simetrizamos
D[np.isinf(D)] = 0.0
D = D + D.T
np.fill_diagonal(D, 0.0)

print(f"matriz DTW: {D.shape}   ({len(SERIES)*(len(SERIES)-1)//2:,} pares)")
print(f"distancia  min/mediana/max: {D[D>0].min():.2f} / {np.median(D[D>0]):.2f} / {D.max():.2f}")

meta = {r["product_id"]: r for r in prod.select("product_id","cat1","cat2","cat3","brand").to_dicts()}
idx = {p: i for i, p in enumerate(PIDS)}

# Vecino mas cercano de cada producto: control de sanidad cualitativo
vecinos = []
for i, p in enumerate(PIDS[:12]):
    j = int(np.argsort(D[i])[1])
    vecinos.append({"producto": p, "vecino": PIDS[j], "dtw": round(float(D[i, j]), 2),
                    "misma_cat3": meta.get(p,{}).get("cat3") == meta.get(PIDS[j],{}).get("cat3"),
                    "cat3": meta.get(p,{}).get("cat3")})
print("\n=== VECINO MAS CERCANO (top 12 por volumen) ===")
print(pl.DataFrame(vecinos))

## 5 — Cuántos clusters, y con qué linkage

Dos decisiones que se toman juntas, porque interactúan.

**El linkage importa más que el k.** Con `average`, la jerarquía va desprendiendo
outliers de a uno: da el silhouette más alto de todos, pero porque separa 22
productos raros del resto y deja 785 juntos. Un silhouette alto sobre una partición
inútil. `ward` reparte de forma pareja en todo el rango de k. La celda de abajo
compara los tres y lo muestra con números, en vez de pedirte que confíes.

**El k se elige con una guarda de balance**: se toma el mejor silhouette **entre los
k cuyo cluster más chico supera `min_frac_cluster`**. Sin esa condición, el óptimo
siempre es "un outlier contra todo lo demás".

Y dos métricas en vez de una, porque acá el silhouette solo engaña:

- **silhouette** — separación estándar, pero premia las particiones degeneradas.
- **ratio intra/entre** — distancia DTW media dentro del grupo dividida por la de
  entre grupos. Más interpretable: `0,85` significa que dos productos del mismo
  cluster están un 15% más cerca que dos cualesquiera. Cerca de `1` = el grupo no
  dice nada.

### Por qué `k` se fija a mano

Conviene decirlo sin vueltas: **en estos datos no hay una estructura de clusters
nítida**. El silhouette de las particiones balanceadas se mueve entre 0,04 y 0,08 —
valores bajos — y **decrece de forma monótona con `k`**. Un criterio de "maximizar
silhouette" devuelve siempre `k=2`, y a `k=2` el ratio intra/entre da 0,934: el mismo
que `cat3`, o sea, ninguna ganancia.

Tiene sentido: son series mensuales de consumo masivo dominadas por una
estacionalidad común. Normalizadas, casi todas se parecen.

Entonces el cluster no es un descubrimiento de grupos naturales, es una
**segmentación blanda** — y la pregunta correcta no es "¿cuál es el k óptimo?" sino
**"¿mejora el WAPE?"**. Por eso `k` es una palanca y el veredicto lo da el
leaderboard de `03_Optuna`, no esta celda. El default `k=6` es un punto de partida
razonable: balanceado, con ratio 0,86 (contra 0,93 de `cat3`) y suficiente
granularidad para que el árbol la aproveche sin fragmentar los datos.

In [ ]:
Dc = squareform(D, checks=False)
N  = len(PIDS)
MIN_TAM = int(PARAM['min_frac_cluster'] * N)
iu = np.triu_indices(N, k=1)
d_iu = D[iu]

def ratio_intra_entre(lab):
    mismo = lab[iu[0]] == lab[iu[1]]
    if mismo.sum() == 0 or (~mismo).sum() == 0:
        return np.nan
    return float(d_iu[mismo].mean() / d_iu[~mismo].mean())

filas = []
for met in ("ward", "complete", "average"):
    Zm = linkage(Dc, method=met)
    for k in range(PARAM['k_min'], PARAM['k_max'] + 1):
        lab = fcluster(Zm, k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        tam = np.bincount(lab)[1:]
        filas.append({
            "linkage": met, "k": k,
            "silhouette": round(float(silhouette_score(D, lab, metric="precomputed")), 4),
            "ratio_intra_entre": round(ratio_intra_entre(lab), 3),
            "min_cluster": int(tam.min()), "max_cluster": int(tam.max()),
            "balanceado": bool(tam.min() >= MIN_TAM),
        })

comp = pl.DataFrame(filas)
print(f"tamanio minimo exigido: {MIN_TAM} productos ({100*PARAM['min_frac_cluster']:.0f}% de {N})\n")
for met in ("ward", "complete", "average"):
    print(comp.filter(pl.col("linkage") == met))

In [ ]:
# Grafico: silhouette por k y por linkage. Tres series -> los tres primeros slots,
# que son los que validan all-pairs. Cada linea va etiquetada al final.
fig, ax = plt.subplots(figsize=(8, 3.6))
for j, met in enumerate(("ward", "complete", "average")):
    g = comp.filter(pl.col("linkage") == met).sort("k")
    ax.plot(g["k"], g["silhouette"], color=SERIE[j], linewidth=2, marker="o", markersize=5)
    bal = g.filter(pl.col("balanceado"))
    if bal.height:      # marca hueca sobre los k que NO cumplen la guarda
        no = g.filter(~pl.col("balanceado"))
        ax.scatter(no["k"], no["silhouette"], s=60, facecolors=FONDO,
                   edgecolors=SERIE[j], linewidths=1.5, zorder=3)
    ax.annotate(met, (g["k"][-1], g["silhouette"][-1]), xytext=(6, 0),
                textcoords="offset points", color=SERIE[j], fontsize=9, va="center")
limpiar(ax, "Silhouette por k · marca hueca = deja algún cluster demasiado chico",
        y="silhouette", x="k")
ax.set_xlim(PARAM['k_min'] - .3, PARAM['k_max'] + 1.4)
plt.tight_layout(); plt.show()

print("Que 'average' gane en silhouette y pierda en balance es exactamente el problema:")
print("separa un punado de outliers y deja todo lo demas en un solo cluster.")

In [ ]:
cand = comp.filter((pl.col("linkage") == PARAM['linkage']) & pl.col("balanceado"))
if cand.height == 0:
    raise RuntimeError(
        f"Ningun k entre {PARAM['k_min']} y {PARAM['k_max']} deja clusters de al menos "
        f"{MIN_TAM} productos con linkage={PARAM['linkage']!r}. "
        f"Baja 'min_frac_cluster' o cambia de linkage."
    )

sugerido = int(cand.sort("silhouette", descending=True)["k"][0])
print(f"k que maximiza silhouette entre los balanceados: {sugerido}")
print(f"   ratio intra/entre en ese k: "
      f"{cand.filter(pl.col('k')==sugerido)['ratio_intra_entre'][0]}")
print(f"   ratio intra/entre en el k mas alto probado: "
      f"{cand.sort('k')['ratio_intra_entre'][-1]}  (k={cand.sort('k')['k'][-1]})")
print()
print("El silhouette decrece con k y el ratio mejora: no hay un optimo interno.")
print("Por eso k lo fija PARAM['k'] y el veredicto lo da el WAPE del leaderboard.")

mejor = PARAM['k'] if PARAM['k'] is not None else sugerido
if PARAM['k'] is not None and not cand.filter(pl.col("k") == PARAM['k']).height:
    raise RuntimeError(
        f"k={PARAM['k']} no esta entre los balanceados para linkage={PARAM['linkage']!r}. "
        f"Balanceados: {sorted(cand['k'].to_list())}"
    )
print(f"\nk elegido = {mejor}" + ("  (de PARAM['k'])" if PARAM['k'] is not None else "  (sugerido)"))

In [ ]:
# K definitivo. Arranca en el sugerido; cambialo a mano si preferis otro
# (mira el ratio_intra_entre de la tabla: suele seguir bajando con k).
K = int(mejor)
Z = linkage(Dc, method=PARAM['linkage'])
labels = fcluster(Z, K, criterion="maxclust")
print(f"K = {K}   linkage = {PARAM['linkage']}")
print(f"ratio intra/entre = {ratio_intra_entre(labels):.3f}")

clus = pl.DataFrame({"product_id": PIDS, "cluster": labels.astype(int)})
resumen = (clus.join(vida.select("product_id","tn_total"), on="product_id", how="left")
               .group_by("cluster")
               .agg(pl.len().alias("n_productos"),
                    pl.col("tn_total").sum().alias("tn"),
                    pl.col("tn_total").median().alias("tn_mediana_producto"))
               .with_columns((100*pl.col("tn")/pl.col("tn").sum()).round(1).alias("%_tn"))
               .sort("cluster"))
print(resumen)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
dendrogram(Z, no_labels=True, color_threshold=Z[-(K-1), 2],
           above_threshold_color=MUDO, ax=ax)
ax.set_title(f"Dendrograma ({PARAM['linkage']} linkage sobre distancia DTW) · corte en k={K}",
             color=TINTA, loc="left", pad=10)
ax.grid(False); ax.set_ylabel("distancia DTW")
plt.tight_layout(); plt.show()

## 6 — Cómo es cada cluster

In [ ]:
# Centroide: mediana punto a punto de las series del cluster, alineadas por indice.
# Con eje='edad' esto es la curva de vida tipica del arquetipo.
LARGO = int(np.median(largos))
ncol = min(4, K); nfil = int(np.ceil(K / ncol))
fig, axes = plt.subplots(nfil, ncol, figsize=(3.4*ncol, 2.5*nfil), squeeze=False)

for c in range(1, K + 1):
    ax = axes[(c-1)//ncol][(c-1)%ncol]
    miembros = [SERIES[i] for i in range(len(PIDS)) if labels[i] == c]
    M = np.full((len(miembros), LARGO), np.nan)
    for r, s in enumerate(miembros):
        M[r, :min(len(s), LARGO)] = s[:LARGO]
    med = np.nanmedian(M, axis=0)
    q25 = np.nanpercentile(M, 25, axis=0); q75 = np.nanpercentile(M, 75, axis=0)
    t = np.arange(LARGO)
    ax.fill_between(t, q25, q75, color=SERIE[0], alpha=.15)
    ax.plot(t, med, color=SERIE[0], linewidth=2)
    ax.axhline(0 if PARAM['normalizacion'] == "zscore" else .5, color=EJE_C, linewidth=.8)
    ax.set_title(f"cluster {c}  ·  n={len(miembros)}", color=TINTA, loc="left", fontsize=9)
    ax.grid(axis="x", visible=False); ax.tick_params(labelsize=7)

for j in range(K, nfil*ncol):
    axes[j//ncol][j%ncol].axis("off")
eje_lbl = "meses (calendario)" if PARAM['eje'] == "calendario" else "edad del producto (meses)"
fig.suptitle(f"Forma típica de cada cluster  ·  eje: {eje_lbl}", color=TINTA, x=.01, ha="left", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Composicion por cat3: heatmap. Magnitud -> rampa secuencial de un solo tono.
from matplotlib.colors import LinearSegmentedColormap
CMAP = LinearSegmentedColormap.from_list("azul", SEQ)

cl = clus.join(prod.select("product_id","cat3"), on="product_id", how="left")
top_cats = (cl.group_by("cat3").len().sort("len", descending=True).head(12)["cat3"].to_list())
cl = cl.with_columns(pl.when(pl.col("cat3").is_in(top_cats)).then(pl.col("cat3"))
                       .otherwise(pl.lit("Otros")).alias("cat3g"))

piv = (cl.group_by(["cluster","cat3g"]).len()
         .pivot(on="cat3g", index="cluster", values="len").fill_null(0).sort("cluster"))
cats = [c for c in piv.columns if c != "cluster"]
A = piv.select(cats).to_numpy().astype(float)
A = 100 * A / np.maximum(A.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(1.0*len(cats)+2.5, .55*K+2))
im = ax.imshow(A, cmap=CMAP, aspect="auto", vmin=0, vmax=A.max())
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(K)); ax.set_yticklabels([f"cluster {c}" for c in piv['cluster']], fontsize=8)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        if A[i, j] >= 5:
            ax.text(j, i, f"{A[i,j]:.0f}", ha="center", va="center", fontsize=7,
                    color="#ffffff" if A[i, j] > A.max()*.55 else TINTA)
ax.grid(False)
ax.set_title("Composición de cada cluster por cat3 (% de productos)", color=TINTA, loc="left", pad=10)
plt.tight_layout(); plt.show()

## 7 — ¿El cluster aporta más que `cat3`?

La prueba directa: silhouette sobre **la misma matriz DTW**, comparando dos
etiquetados — el del cluster y el de `cat3`. Si `cat3` agrupara bien por
comportamiento, su silhouette sería parecido.

Está sesgado a favor del cluster (se optimizó justamente para maximizar esa
métrica), así que el número que importa no es cuánto gana, sino **cuánto se aleja
de cero `cat3`**: un silhouette de `cat3` cercano a 0 significa que la jerarquía
no separa comportamientos en absoluto.

In [ ]:
cat3_lab = np.array([str(meta.get(p, {}).get("cat3", "NA")) for p in PIDS])
_, cat3_num = np.unique(cat3_lab, return_inverse=True)
brand_lab = np.array([str(meta.get(p, {}).get("brand", "NA")) for p in PIDS])
_, brand_num = np.unique(brand_lab, return_inverse=True)

def sil(lab):
    return float(silhouette_score(D, lab, metric="precomputed")) if len(set(lab)) > 1 else float("nan")

print(f"silhouette  cluster DTW (k={K}) : {sil(labels):+.4f}   <- optimizado para esto")
print(f"silhouette  cat3  ({len(set(cat3_num))} grupos) : {sil(cat3_num):+.4f}")
print(f"silhouette  brand ({len(set(brand_num))} grupos): {sil(brand_num):+.4f}")
print()
print("Cerca de 0 = los grupos no separan comportamiento (los productos de un grupo")
print("no se parecen mas entre si que a los de otro). Negativo = agrupa mal.")

# Ratio intra/entre, mas interpretable que el silhouette
r_clu = ratio_intra_entre(labels)
for nombre, lab in [(f"cluster DTW k={K}", labels), ("cat3", cat3_num), ("brand", brand_num)]:
    print(f"{nombre:18s}  ratio DTW intra/entre = {ratio_intra_entre(lab):.3f}")
r_cat = ratio_intra_entre(cat3_num)
print(f"\nEl cluster agrupa {100*(r_cat-r_clu)/r_cat:.0f}% mas compacto que cat3.")
print("Si esa diferencia es chica, la feature probablemente no mueva el WAPE:")
print("comprobalo en el leaderboard antes de darla por buena.")
print("\nRatio bajo = grupos compactos. Ratio ~1 = el grupo no dice nada.")

## 8 — Exportar la feature

In [ ]:
salida = (clus
    .join(prod.select("product_id","cat1","cat2","cat3","brand"), on="product_id", how="left")
    .join(vida.select("product_id","tn_total","m_nace","m_muere"), on="product_id", how="left")
    .with_columns(
        pl.lit(PARAM['eje']).alias("dtw_eje"),
        pl.lit(PARAM['mes_corte']).alias("dtw_mes_corte"),
        pl.lit(K).alias("dtw_k"),
    )
    .select("product_id", "cluster", "dtw_eje", "dtw_mes_corte", "dtw_k",
            "cat1", "cat2", "cat3", "brand", "tn_total")
    .rename({"cluster": "cluster_dtw"}))

sufijo = f"{PARAM['eje']}_k{K}_corte{PARAM['mes_corte']}"
out = DIR_OUT / f"clusters_dtw_{sufijo}.parquet"
salida.write_parquet(out)

print(f"Guardado: {out}")
print(f"{salida.height} productos etiquetados\n")
print(salida.head(8))

faltan = set(prod["product_id"].to_list()) - set(salida["product_id"].to_list())
print(f"\nProductos SIN cluster (menos de {PARAM['min_meses']} meses de historia): {len(faltan)}")
print("Al pegar la feature quedan null -> conviene una categoria explicita 'sin_cluster',")
print("porque 'poca historia' es en si misma una senial.")

### Cómo usarlo en `02_FE`

Pegalo por `product_id` y declaralo como categórica:

```python
clusters = pl.read_parquet(RUTA_FE / "clusters_dtw_calendario_k6_corte201906.parquet")
df_norm = df_norm.join(clusters.select("product_id", "cluster_dtw"),
                       on="product_id", how="left")
df_norm = df_norm.with_columns(pl.col("cluster_dtw").fill_null(-1))   # -1 = sin cluster
```

Y en `03_Optuna`, agregala a las categóricas:

```python
'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand', 'cluster_dtw'],
```

Después comparás en el leaderboard contra el experimento sin cluster. Como el nombre
del experimento no cambia por agregar una columna al dataset, usá la palanca `sufijo`
para distinguirlos: `'sufijo': 'conClusterDTW'`.

### Feature o modelo por cluster

| | a favor | en contra |
|---|---|---|
| **Cluster como feature** | un solo modelo, todos los datos, el árbol decide si usarla | la interacción cluster×resto queda librada al modelo |
| **Un modelo por cluster** | cada grupo tiene sus propios splits e hiperparámetros | divide los datos: un cluster de 40 productos entrena con muy poco |

Con estos tamaños de cluster, **empezá por la feature**. Un modelo por cluster recién
tiene sentido si algún cluster concentra volumen suficiente para sostenerlo solo —
mirá la columna `%_tn` de la tabla de la sección 5. La tabla de abajo lo estima.

In [ ]:
est = (resumen.with_columns(
        (pl.col("n_productos") * 30).alias("filas_aprox_si_modelo_propio"),
        pl.when(pl.col("n_productos") >= 150).then(pl.lit("si"))
          .when(pl.col("n_productos") >= 60).then(pl.lit("dudoso"))
          .otherwise(pl.lit("no")).alias("modelo_propio_viable"))
       .sort("%_tn", descending=True))
print(est)
print("\nRegla usada: >=150 productos viable, 60-150 dudoso, <60 no.")
print("Es orientativa: lo que decide es cuantas FILAS de entrenamiento quedan,")
print("y eso depende de la granularidad (con producto-cliente hay muchas mas).")

## Qué probar después

1. **Correr con `eje='edad'`** y comparar los arquetipos: ahí los clusters describen
   *formas de ciclo de vida* (el boom que se apaga, el estable, el que crece lento),
   que es más interpretable para negocio aunque suele servir menos como feature.
2. **Mover `window`**: con `window=2` sólo agrupa series casi sincronizadas; con
   `window=12` tolera desfases de un año y todo empieza a parecerse.
3. **Verificar el corte**: si cambiás los meses de train en `03_Optuna`, hay que
   regenerar los clusters con el `mes_corte` nuevo. El nombre del parquet lo lleva
   incluido justamente para no confundirse.